# Paper 1 – Revision analyses (reviewer points C, D, E)

Uses your saved Paper 1 results and models; reads OASIS-1 only. Run with a **T4 GPU**.
- Cells 1–5: setup and data (same as the Paper 1 notebook)
- **R1** confusion matrices + cumulative AUCs (no training, ~2 min)
- **R2** region-occlusion control (saved models, no training, ~10 min)
- **R3** repeated cross-validation, 2 extra seeds (training, ~3–4 h, resumes after a disconnect)
- **R4** age as a confounder: age-only baseline, 70–85 age band, score-age correlation (no training, ~1 min)
- **R5** re-draw Figure 4 (average Grad-CAM per stage) at 600 dpi (saved models, no training, ~5–10 min)


In [ ]:
!pip -q install nibabel openpyxl
# 1. Setup
import os, glob, json, gc, numpy as np, pandas as pd, nibabel as nib, cv2, psutil
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass
print('TensorFlow', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
# 2. Settings  (change only if you know why)
DATA     = '/content/drive/MyDrive/OASIS1_masked'     # folder with the OASIS-1 .img/.hdr files + the OASIS .xlsx
RESULTS  = '/content/drive/MyDrive/OASIS1_results'    # shared with the ICCIDE notebook (slice cache lives here)
OUT      = f'{RESULTS}/paper1'                        # Paper 1 results are saved here
IMG      = 128      # slice size
N_SLICES = 25       # coronal slices per subject
STEP     = 2        # every 2nd slice
N_FOLDS  = 5
SEED     = 42
BATCH    = 32
EPOCHS   = 60       # maximum epochs (early stopping usually ends sooner)
N_CLS    = 3        # 0 = NonDemented, 1 = Very Mild, 2 = Mild/Moderate
os.makedirs(OUT, exist_ok=True)

In [ ]:
# 3. Labels: keep only MR1 sessions that have a CDR score and an image file
xlsx = [f for f in os.listdir(DATA) if f.endswith(('.xlsx', '.csv'))][0]
df = pd.read_excel(f'{DATA}/{xlsx}') if xlsx.endswith('.xlsx') else pd.read_csv(f'{DATA}/{xlsx}')
df = df[df['ID'].astype(str).str.contains('_MR1') & df['CDR'].notna()].copy()

def find_img(sid):
    hits = glob.glob(f'{DATA}/{sid}_mpr_n*_anon_111_t88_masked_gfc.img')
    return hits[0] if hits else None

df['path']  = df['ID'].map(find_img)
df = df[df['path'].notna()].sort_values('ID').reset_index(drop=True)
df['label'] = (df['CDR'] > 0).astype(int)          # 0 = NonDemented (CDR 0), 1 = Demented (CDR > 0)
ids, y = df['ID'].values, df['label'].values
print(len(df), 'subjects |', (y == 0).sum(), 'NonDemented |', (y == 1).sum(), 'Demented')
print(df['CDR'].value_counts().sort_index().to_string())
assert len(set(ids)) == len(ids), 'duplicate subjects!'
df[['ID', 'Age', 'CDR', 'label']].to_csv(f'{RESULTS}/cohort.csv', index=False)

In [ ]:
# 4. Load / extract coronal slices - memory-safe version (saves progress every 40 subjects; if the session crashes, just run again and it continues)
import gc, psutil

def load_volume(path):
    img = nib.load(path, mmap=False)                           # plain file read (no memory-mapping of Drive files)
    v = np.squeeze(np.asarray(img.dataobj)).astype(np.float32)
    del img
    v /= np.percentile(v[v > 0], 99.5)                         # per-volume intensity normalisation to [0, 1]
    np.clip(v, 0, 1, out=v)
    return v

def coronal_slices(v, centre):
    ap = int(np.argmax(v.shape))                               # the 208-voxel axis is anterior-posterior
    out = []
    for k in range(N_SLICES):
        idx = centre + (k - N_SLICES // 2) * STEP
        sl = np.ascontiguousarray(np.rot90(np.take(v, idx, axis=ap)))   # rows = inferior-superior, cols = left-right
        out.append(cv2.resize(sl, (IMG, IMG), interpolation=cv2.INTER_AREA))
    return np.stack(out).astype(np.float16)

CACHE = f'{RESULTS}/slices_{IMG}_{N_SLICES}x{STEP}.npz'
CHUNK = 40
part_file = lambda s: f'{RESULTS}/slices_part_{s:03d}.npz'

if os.path.exists(CACHE):
    z = np.load(CACHE, allow_pickle=True)
    assert list(z['ids']) == list(ids), 'cache does not match cohort - delete the .npz file and re-run'
    X, CENTRE = z['X'], int(z['centre'])
else:
    cfile = f'{RESULTS}/centre.txt'
    if os.path.exists(cfile):
        CENTRE = int(open(cfile).read())
    else:   # all volumes share one atlas, so one position fits everyone: centre of the brain's front-to-back extent
        cs = []
        for p in df['path'][:10]:
            v = load_volume(p); ap = int(np.argmax(v.shape))
            nzv = np.where(v.sum(axis=tuple(a for a in range(3) if a != ap)) > 0)[0]
            cs.append((nzv[0] + nzv[-1]) // 2); del v
        CENTRE = int(np.median(cs)); open(cfile, 'w').write(str(CENTRE))
    for s in range(0, len(df), CHUNK):
        if os.path.exists(part_file(s)): continue
        part = np.stack([coronal_slices(load_volume(p), CENTRE) for p in df['path'][s:s + CHUNK]])
        np.savez_compressed(part_file(s), X=part, ids=ids[s:s + CHUNK])
        del part; gc.collect()
        print(f'{min(s + CHUNK, len(df))}/{len(df)} subjects done | RAM in use: {psutil.virtual_memory().used / 1e9:.1f} GB')
    parts = [np.load(part_file(s), allow_pickle=True) for s in range(0, len(df), CHUNK)]
    X = np.concatenate([p['X'] for p in parts])
    assert list(np.concatenate([p['ids'] for p in parts])) == list(ids)
    np.savez_compressed(CACHE, X=X, ids=ids, centre=CENTRE)
    for s in range(0, len(df), CHUNK): os.remove(part_file(s))
print('slices:', X.shape, '| centre coronal index:', CENTRE, '| RAM in use: %.1f GB' % (psutil.virtual_memory().used / 1e9))

In [ ]:
# 5. Three-stage labels from CDR
cdr = df['CDR'].astype(float).values
stage = np.select([cdr == 0, cdr == 0.5, cdr >= 1], [0, 1, 2])   # Moderate (CDR 2, n = 2) is merged with Mild
print('3-stage counts [NonDemented, VeryMild, Mild/Moderate]:', np.bincount(stage))
np.save(f'{OUT}/stage3_labels.npy', stage)

In [ ]:
# 6. Shared helpers
import json
import scipy.ndimage as ndi
from sklearn.metrics import confusion_matrix, roc_auc_score, cohen_kappa_score, f1_score
NS = X.shape[1]
NAMES = ['NonDemented', 'VeryMild', 'Mild/Moderate']
mf1 = lambda y, p: f1_score(y, p, average='macro', zero_division=0)
qwk = lambda y, p: cohen_kappa_score(y, p, weights='quadratic')
to_ord = lambda y: np.stack([y > 0, y > 1], axis=1).astype('float32')

def apply_cuts(s, t1, t2):
    return np.where(s < t1, 0, np.where(s < t2, 1, 2))

def tune(s, y, metric=mf1):
    grid, best = np.arange(0.10, 1.95, 0.05), (0.5, 1.5, -1)
    for t1 in grid:
        for t2 in grid[grid > t1]:
            v = metric(y, apply_cuts(s, t1, t2))
            if v > best[2]: best = (t1, t2, v)
    return best[:2]

def folds(seed):
    out = []
    for tr, te in StratifiedKFold(N_FOLDS, shuffle=True, random_state=seed).split(np.zeros(len(stage)), stage):
        tr, va = train_test_split(tr, test_size=0.15, stratify=stage[tr], random_state=seed)
        out.append((tr, va, te))
    return out

def summary(y, p):
    cm = confusion_matrix(y, p, labels=[0, 1, 2]); rec = cm.diagonal() / np.maximum(cm.sum(axis=1), 1)
    return {'acc': accuracy_score(y, p), 'bal_acc': balanced_accuracy_score(y, p), 'macro_F1': mf1(y, p),
            'QWK': qwk(y, p), 'ND_rec': rec[0], 'VM_rec': rec[1], 'MM_rec': rec[2]}
print('Helpers ready')

In [ ]:
# R1. Confusion matrices and cumulative AUCs for the three main models (no training)
def exp1_preds():                       # standard CNN: averaged softmax probabilities per subject
    p, s0, s1 = np.full(len(stage), -1), np.zeros(len(stage)), np.zeros(len(stage))
    for k in range(N_FOLDS):
        r = json.load(open(f'{OUT}/exp1/subject_fold{k}.json')); P = np.array(r['probs']); i = r['test_idx']
        p[i], s0[i], s1[i] = P.argmax(1), P[:, 1] + P[:, 2], P[:, 2]
    return p, s0, s1

def tuned_preds(E, metric):             # ordinal models: disease score + cut-offs tuned on validation subjects
    p, s = np.full(len(stage), -1), np.zeros(len(stage))
    for k in range(N_FOLDS):
        r = json.load(open(f'{E}/fold{k}.json'))
        t1, t2 = tune(np.array(r['val_score']), stage[r['val_idx']], metric)
        p[r['test_idx']], s[r['test_idx']] = apply_cuts(np.array(r['score']), t1, t2), np.array(r['score'])
    return p, s, s

p1, a1, b1 = exp1_preds()
p2, a2, b2 = tuned_preds(f'{OUT}/exp3_ordinal_tuned', mf1)
p3, a3, b3 = tuned_preds(f'{OUT}/exp4_densenet_ordinal', balanced_accuracy_score)
rng = np.random.default_rng(SEED); boot = [rng.integers(0, len(stage), len(stage)) for _ in range(2000)]
y = stage
rows = []
for name, p, s0, s1 in [('Standard CNN', p1, a1, b1), ('Ordinal Custom CNN (final)', p2, a2, b2), ('DenseNet121 ordinal', p3, a3, b3)]:
    print(f'\n===== {name} =====')
    print(pd.DataFrame(confusion_matrix(y, p, labels=[0, 1, 2]), index=NAMES, columns=NAMES))
    auc0, auc1 = roc_auc_score(y > 0, s0), roc_auc_score(y > 1, s1)
    ci0 = np.percentile([roc_auc_score(y[i] > 0, s0[i]) for i in boot], [2.5, 97.5])
    ci1 = np.percentile([roc_auc_score(y[i] > 1, s1[i]) for i in boot], [2.5, 97.5])
    rows.append({'model': name, 'AUC stage>0 (ND vs rest)': f'{auc0:.3f} [{ci0[0]:.3f}, {ci0[1]:.3f}]',
                 'AUC stage>1 (MM vs rest)': f'{auc1:.3f} [{ci1[0]:.3f}, {ci1[1]:.3f}]'})
print('\n===== Cumulative AUCs [95% bootstrap CI] =====')
print(pd.DataFrame(rows).to_string(index=False))
pd.DataFrame(rows).to_csv(f'{OUT}/revision_R1_auc.csv', index=False)

In [ ]:
# R2. Region-occlusion control (saved final models, no training)
#     Each region is altered on every test slice and the change in predictions is measured:
#       core  -> set to 0 (removes brain tissue)
#       CSF   -> filled with the median brain intensity of the slice (removes fluid contrast)
#       edge  -> filled with the median brain intensity (removes the gap at the brain surface)
#       far   -> random noise U(0, 0.3) in the far background (background control)
E5 = f'{OUT}/exp5_gradcam'

def brain_mask(sl):
    m = ndi.binary_fill_holes(ndi.binary_closing(sl > 0.10, iterations=2))
    lab, n = ndi.label(m)
    return lab == (np.argmax(np.bincount(lab.ravel())[1:]) + 1) if n > 1 else m

def regions(sl):
    mask = brain_mask(sl)
    ero, dil = ndi.binary_erosion(mask, iterations=4), ndi.binary_dilation(mask, iterations=4)
    csf = ero & (sl < 0.25)
    return {'core': ero & ~csf, 'CSF': csf, 'edge': dil & ~ero, 'far': ~dil}, mask

def occlude(vol, cond, rng):
    out = vol.copy()
    for j in range(vol.shape[0]):
        sl = vol[j]; reg, mask = regions(sl); med = float(np.median(sl[mask])) if mask.any() else 0.0
        if cond == 'core': out[j][reg['core']] = 0
        elif cond == 'CSF': out[j][reg['CSF']] = med
        elif cond == 'edge': out[j][reg['edge']] = med
        elif cond == 'far': out[j][reg['far']] = rng.uniform(0, 0.3, reg['far'].sum())
    return out

def score(model, vols):
    P = model.predict(vols.reshape(-1, IMG, IMG, 1), batch_size=256, verbose=0)
    return P.reshape(len(vols), NS, 2).sum(axis=2).mean(axis=1)

CONDS = ['none', 'core', 'CSF', 'edge', 'far']
sc = {c: np.zeros(len(stage)) for c in CONDS}; pr = {c: np.full(len(stage), -1) for c in CONDS}
rng = np.random.default_rng(SEED)
for k, (tr, va, te) in enumerate(folds(SEED)):                       # same folds as the final model
    m = keras.models.load_model(f'{E5}/model_fold{k}.keras')
    t1, t2 = tune(score(m, X[va].astype('float32')), stage[va])        # same rule as the final model
    vols = X[te].astype('float32')
    for c in CONDS:
        v = vols if c == 'none' else np.stack([occlude(vol, c, rng) for vol in vols])
        s = score(m, v); sc[c][te] = s; pr[c][te] = apply_cuts(s, t1, t2)
    print(f'fold {k} done')

rows = []
for c in CONDS:
    d = sc[c] - sc['none']
    rows.append({'condition': c, 'bal_acc': balanced_accuracy_score(stage, pr[c]), 'macro_F1': mf1(stage, pr[c]),
                 'mean |change in score|': np.abs(d).mean(),
                 **{f'mean change {NAMES[s]}': d[stage == s].mean() for s in range(3)}})
t = pd.DataFrame(rows).round(3); t.to_csv(f'{OUT}/revision_R2_occlusion.csv', index=False)
pd.set_option('display.width', 250)
print('\n===== Region occlusion (disease score 0-2; negative change = predicted LESS severe) =====')
print(t.to_string(index=False))

In [ ]:
# R3. Repeated cross-validation: 2 extra seeds for the standard CNN and the final ordinal CNN (~3-4 h; resumes)
R3 = f'{OUT}/revision_R3_repeats'; os.makedirs(R3, exist_ok=True)
EXTRA_SEEDS = [1, 2]

def body(x):
    for f in (32, 64, 128):
        x = layers.Conv2D(f, 3, padding='same', use_bias=False, kernel_initializer='he_normal')(x)
        x = layers.BatchNormalization()(x); x = layers.ReLU()(x)
        x = layers.MaxPooling2D()(x); x = layers.Dropout(0.1)(x)
    x = layers.Flatten()(x)
    x = layers.Dense(64, use_bias=False, kernel_initializer='he_normal')(x)
    x = layers.BatchNormalization()(x); x = layers.ReLU()(x)
    return layers.Dropout(0.5)(x)

def build(kind):
    inp = keras.Input((IMG, IMG, 1))
    x = layers.RandomFlip('horizontal')(inp); x = layers.RandomRotation(0.03)(x); x = layers.RandomTranslation(0.05, 0.05)(x)
    x = body(x)
    if kind == 'standard':
        out = layers.Dense(N_CLS, activation='softmax')(x); loss = 'sparse_categorical_crossentropy'
    else:
        out = layers.Dense(N_CLS - 1, activation='sigmoid')(x); loss = 'binary_crossentropy'
    m = keras.Model(inp, out); m.compile(optimizer=keras.optimizers.Adam(3e-4, clipnorm=1.0), loss=loss); return m

def subj(model, idx, kind):
    P = model.predict(X[idx].reshape(-1, IMG, IMG, 1).astype('float32'), batch_size=256, verbose=0)
    return P.reshape(len(idx), NS, -1).mean(axis=1)                  # mean over slices

for seed in EXTRA_SEEDS:
    for kind in ['standard', 'ordinal']:
        for k, (tr, va, te) in enumerate(folds(seed)):
            f = f'{R3}/{kind}_seed{seed}_fold{k}.json'
            if os.path.exists(f): print(kind, 'seed', seed, 'fold', k, 'done before'); continue
            Xtr, ytr = X[tr].reshape(-1, IMG, IMG, 1).astype('float32'), np.repeat(stage[tr], NS)
            Xva, yva = X[va].reshape(-1, IMG, IMG, 1).astype('float32'), np.repeat(stage[va], NS)
            cw = compute_class_weight('balanced', classes=np.arange(N_CLS), y=ytr)
            keras.backend.clear_session(); tf.random.set_seed(seed)
            m = build(kind)
            cb = [keras.callbacks.EarlyStopping('val_loss', patience=10, restore_best_weights=True),
                  keras.callbacks.ReduceLROnPlateau('val_loss', factor=0.5, patience=4, min_lr=1e-6)]
            if kind == 'standard':
                m.fit(Xtr, ytr, validation_data=(Xva, yva), class_weight=dict(enumerate(cw)),
                      epochs=EPOCHS, batch_size=BATCH, callbacks=cb, verbose=0)
            else:
                m.fit(Xtr, to_ord(ytr), sample_weight=cw[ytr], validation_data=(Xva, to_ord(yva), cw[yva]),
                      epochs=EPOCHS, batch_size=BATCH, callbacks=cb, verbose=0)
            del Xtr, Xva
            json.dump({'test_idx': te.tolist(), 'val_idx': va.tolist(),
                       'test_out': subj(m, te, kind).tolist(), 'val_out': subj(m, va, kind).tolist()}, open(f, 'w'))
            print(kind, 'seed', seed, 'fold', k, 'saved')

# ---- Summary over 3 repeats (seed 42 = original Experiments 1 and 3) ----
def pooled(kind, seed):
    p = np.full(len(stage), -1)
    for k in range(N_FOLDS):
        if seed == SEED:
            if kind == 'standard':
                r = json.load(open(f'{OUT}/exp1/subject_fold{k}.json')); p[r['test_idx']] = np.argmax(r['probs'], 1); continue
            r = json.load(open(f'{OUT}/exp3_ordinal_tuned/fold{k}.json')); s_te, s_va = np.array(r['score']), np.array(r['val_score'])
        else:
            r = json.load(open(f'{R3}/{kind}_seed{seed}_fold{k}.json'))
            if kind == 'standard': p[r['test_idx']] = np.argmax(r['test_out'], 1); continue
            s_te, s_va = np.array(r['test_out']).sum(1), np.array(r['val_out']).sum(1)
        t1, t2 = tune(s_va, stage[r['val_idx']]); p[r['test_idx']] = apply_cuts(s_te, t1, t2)
    return p

rows = []
for seed in [SEED] + EXTRA_SEEDS:
    for kind in ['standard', 'ordinal']:
        rows.append({'seed': seed, 'model': kind, **summary(stage, pooled(kind, seed))})
res = pd.DataFrame(rows); res.to_csv(f'{R3}/summary.csv', index=False)
pd.set_option('display.width', 250)
print('\n===== Per repeat =====');           print(res.round(3).to_string(index=False))
print('\n===== Mean +/- SD over 3 repeats ====='); print(res.drop(columns='seed').groupby('model').agg(['mean', 'std']).round(3).T)
d = res[res.model == 'ordinal'].set_index('seed').bal_acc - res[res.model == 'standard'].set_index('seed').bal_acc
print('\nOrdinal minus standard balanced accuracy per repeat:', d.round(3).to_dict(), '| mean', round(d.mean(), 3))

In [ ]:
# R4. Age as a confounder (no training, ~1 min)
#     Reproduces Table 1 (age and sex) and Section 4.6 of the manuscript:
#       (a) age and sex per stage, with Kruskal-Wallis and chi-square tests
#       (b) an age-only baseline, evaluated with the SAME folds and the same decision rule
#       (c) both models restricted to subjects aged 70-85 years, where the stages differ less in age
#       (d) Spearman correlation between the disease score and age within each stage
from scipy.stats import kruskal, chi2_contingency, spearmanr

age = df['Age'].astype(float).values                      # same order as `stage`
sex = df['M/F'].astype(str).values if 'M/F' in df.columns else None
y = stage

# --- (a) demographics per stage ---
tab = pd.DataFrame({'n': [int((y == s).sum()) for s in range(3)],
                    'age mean': [round(age[y == s].mean(), 1) for s in range(3)],
                    'age SD':   [round(age[y == s].std(ddof=1), 1) for s in range(3)],
                    'age min':  [int(age[y == s].min()) for s in range(3)],
                    'age max':  [int(age[y == s].max()) for s in range(3)]}, index=NAMES)
if sex is not None:
    tab['women %'] = [round((sex[y == s] == 'F').mean() * 100, 1) for s in range(3)]
print('===== Cohort by stage (Table 1) ====='); print(tab.to_string())
print('Age across stages:  Kruskal-Wallis p = %.3g' % kruskal(*[age[y == s] for s in range(3)]).pvalue)
if sex is not None:
    print('Sex across stages:  chi-square   p = %.3g' % chi2_contingency(pd.crosstab(y, sex))[1])
    print('Women overall: %.1f%%' % ((sex == 'F').mean() * 100))

# --- final model predictions, and an age-only baseline on the same folds ---
def tune_grid(v, yv):
    """Two cut-offs on VALIDATION subjects, macro-F1, on a percentile grid of v (works for any scale)."""
    grid, best = np.unique(np.percentile(v, np.arange(2, 99, 2))), (None, None, -1)
    for a in grid:
        for b in grid[grid > a]:
            s = mf1(yv, np.where(v < a, 0, np.where(v < b, 1, 2)))
            if s > best[2]: best = (a, b, s)
    return best[:2]

score = np.full(len(y), np.nan); pred_cnn = np.full(len(y), -1); pred_age = np.full(len(y), -1)
for k in range(N_FOLDS):
    r = json.load(open(f'{OUT}/exp3_ordinal_tuned/fold{k}.json'))
    te, va = r['test_idx'], r['val_idx']
    score[te] = r['score']
    t1, t2 = tune(np.array(r['val_score']), y[va])                 # final model: macro-F1 on validation
    pred_cnn[te] = apply_cuts(np.array(r['score']), t1, t2)
    a, b = tune_grid(age[va], y[va])                               # age-only: same rule, same subjects
    pred_age[te] = np.where(age[te] < a, 0, np.where(age[te] < b, 1, 2))
assert (pred_cnn >= 0).all() and (pred_age >= 0).all()

rng = np.random.default_rng(SEED)
def boot_ci(yy, pp, B=2000):
    idx = [rng.integers(0, len(yy), len(yy)) for _ in range(B)]
    return np.percentile([balanced_accuracy_score(yy[i], pp[i]) for i in idx], [2.5, 97.5])

# --- (b) and (c) ---
band = (age >= 70) & (age <= 85)
print('\n===== Final CNN vs an age-only baseline (same folds, same decision rule) =====')
rows = []
for label, m in [('All subjects', np.ones(len(y), bool)), ('Age 70-85', band)]:
    for name, p in [('CNN (final)', pred_cnn), ('Age only', pred_age)]:
        lo, hi = boot_ci(y[m], p[m]); s = summary(y[m], p[m])
        rows.append({'subset': label, 'n': int(m.sum()), 'model': name,
                     'bal_acc': round(s['bal_acc'], 3), '95% CI': f"[{lo:.3f}, {hi:.3f}]",
                     'macro_F1': round(s['macro_F1'], 3), 'ND_rec': round(s['ND_rec'], 3),
                     'VM_rec': round(s['VM_rec'], 3), 'MM_rec': round(s['MM_rec'], 3)})
age_tab = pd.DataFrame(rows); pd.set_option('display.width', 200)
print(age_tab.to_string(index=False))
print('Composition of the 70-85 band [ND, VM, MM]:', np.bincount(y[band], minlength=3))

# --- (d) disease score vs age within each stage ---
print('\n===== Disease score vs age, within each stage =====')
for s in range(3):
    rho, p = spearmanr(score[y == s], age[y == s])
    print(f'  {NAMES[s]:14s} Spearman rho = {rho:+.2f}, p = {p:.3g}  (n = {int((y == s).sum())})')

age_tab.to_csv(f'{OUT}/revision_R4_age.csv', index=False)
tab.to_csv(f'{OUT}/revision_R4_demographics.csv')
print('\nSaved to', f'{OUT}/revision_R4_age.csv')


In [ ]:
# R5. Re-draw Figure 4 (average Grad-CAM per stage) at publication quality
#     Same final models, same test subjects, same Grad-CAM rule as the paper - only the drawing changes.
#     No training. Runs on CPU (~5-10 min) or GPU (~2 min). Needs cells 1-6 of this notebook.
#     Saves Figure4_gradcam_per_stage_v2.png (600 dpi) and .tif in OUT/figures.
from matplotlib import colormaps
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

E5 = f'{OUT}/exp5_gradcam'
FIG = f'{OUT}/figures'; os.makedirs(FIG, exist_ok=True)
SLICES = [6, 12, 18]                                     # 0-based -> slices 7, 13 and 19 of 25
LABELS = ['Non-Demented', 'Very Mild', 'Mild/Moderate']

def last_act_layer(model):                               # deepest 4-D conv/activation layer (the paper's rule)
    for l in reversed(model.layers):
        if isinstance(l, (layers.Conv2D, layers.ReLU, layers.Activation)) and len(l.output.shape) == 4:
            return l
    raise ValueError('no 4-D conv layer found')

def score_of(model, vols):
    P = model.predict(vols.reshape(-1, IMG, IMG, 1), batch_size=256, verbose=0)
    return P.reshape(len(vols), NS, 2).sum(axis=2).mean(axis=1)

def gradcam(model, x, sign):                             # target = disease score; its negative for Non-Demented
    lay = last_act_layer(model)
    gm = keras.Model(model.inputs, [lay.output, model.output])
    out = []
    for i in range(0, len(x), 128):
        xb = tf.convert_to_tensor(x[i:i + 128]); sb = tf.convert_to_tensor(sign[i:i + 128], tf.float32)
        with tf.GradientTape() as tape:
            conv, pred = gm(xb, training=False)
            loss = tf.reduce_sum(sb * tf.reduce_sum(pred, axis=1))
        g = tape.gradient(loss, conv)
        w = tf.reduce_mean(g, axis=(1, 2), keepdims=True)
        cam = tf.nn.relu(tf.reduce_sum(w * conv, axis=-1, keepdims=True))
        cam = tf.image.resize(cam, (IMG, IMG), method='bilinear')[..., 0].numpy()
        lo, hi = cam.min(axis=(1, 2), keepdims=True), cam.max(axis=(1, 2), keepdims=True)
        out.append((cam - lo) / (hi - lo + 1e-8))        # normalised per slice
    return np.concatenate(out), lay.name, tuple(lay.output.shape[1:3])

sum_cam = np.zeros((3, len(SLICES), IMG, IMG)); sum_img = np.zeros_like(sum_cam); n_ok = np.zeros(3, int)
for k, (tr, va, te) in enumerate(folds(SEED)):           # same folds and cut-off rule as R2
    keras.backend.clear_session()
    m = keras.models.load_model(f'{E5}/model_fold{k}.keras', compile=False)
    t1, t2 = tune(score_of(m, X[va].astype('float32')), stage[va])
    vols = X[te].astype('float32')
    pred = apply_cuts(score_of(m, vols), t1, t2)
    ok = np.where(pred == stage[te])[0]                  # correctly classified test subjects only
    xs = vols[ok][:, SLICES].reshape(-1, IMG, IMG, 1)
    sign = np.repeat(np.where(pred[ok] == 0, -1.0, 1.0), len(SLICES))
    cams, lname, lsize = gradcam(m, xs, sign)
    cams = cams.reshape(len(ok), len(SLICES), IMG, IMG)
    for j, i in enumerate(ok):
        s = stage[te][i]; sum_cam[s] += cams[j]; sum_img[s] += vols[i][SLICES]; n_ok[s] += 1
    print(f'fold {k} done | Grad-CAM layer "{lname}" with a {lsize[0]} x {lsize[1]} map')

avg_cam = sum_cam / n_ok[:, None, None, None]; avg_img = sum_img / n_ok[:, None, None, None]
print('Correctly classified per stage:', dict(zip(LABELS, n_ok.tolist())),
      '| paper Figure 4: Non-Demented 89, Very Mild 37, Mild/Moderate 16')
np.savez_compressed(f'{FIG}/figure4_gradcam_averages.npz', cam=avg_cam, img=avg_img, n=n_ok)

# ---- drawing: grey anatomy + heat map whose transparency follows its value (no blue wash) ----
cmap = colormaps['turbo']
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 9})
fig, ax = plt.subplots(3, 3, figsize=(6.7, 5.6), dpi=600,
                       gridspec_kw=dict(wspace=0.04, hspace=0.06, left=0.12, right=0.86, top=0.94, bottom=0.02))
for s in range(3):
    for c in range(len(SLICES)):
        a = ax[s, c]; img = avg_img[s, c]; cam = avg_cam[s, c] / (avg_cam[s, c].max() + 1e-8)
        rgba = cmap(cam); rgba[..., 3] = np.clip(cam, 0, 1) ** 1.2 * 0.75
        a.imshow(img, cmap='gray', vmin=0, vmax=np.percentile(img, 99.5), interpolation='lanczos')
        a.imshow(rgba, interpolation='bicubic')
        a.set_xticks([]); a.set_yticks([])
        for sp in a.spines.values(): sp.set_visible(False)
        if s == 0: a.set_title(f'Slice {SLICES[c] + 1} of 25', fontsize=9)
    ax[s, 0].set_ylabel(f'{LABELS[s]}\n(n = {n_ok[s]})', fontsize=9, rotation=90, labelpad=6)
cax = fig.add_axes([0.88, 0.22, 0.022, 0.56])
cb = fig.colorbar(ScalarMappable(Normalize(0, 1), cmap), cax=cax)
cb.set_label('Normalised Grad-CAM', fontsize=9); cb.ax.tick_params(labelsize=8)
fig.savefig(f'{FIG}/Figure4_gradcam_per_stage_v2.png', dpi=600)
fig.savefig(f'{FIG}/Figure4_gradcam_per_stage_v2.tif', dpi=600, pil_kwargs={'compression': 'tiff_lzw'})
plt.show()
print('Saved to', FIG)
